# Neural Network Training

In [48]:
# import libraries
from pyspark.sql import SparkSession
import pandas as pd
import unicodedata



In [49]:
# define training data set
data_train = "../data/gold/df_latam_names_for_nn_training/"

In [50]:
# define a spark session
spark = SparkSession.builder \
    .appName("neural_network") \
    .master("local[*]") \
    .getOrCreate()

## 1. Data review and validation

In [51]:
# Read structured data on parquet format from gold stage
df_spark = spark.read.parquet(data_train)
df_spark.show(5)

+-------+------+
|   NAME|GENDER|
+-------+------+
|  Elena|     F|
|   Anyi|     F|
|  Maura|     F|
|Yamilet|     F|
|  Joana|     F|
+-------+------+
only showing top 5 rows


In [52]:
# convert to pandas
df_latam = df_spark.toPandas()
df_latam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 831200 entries, 0 to 831199
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   NAME    831200 non-null  object
 1   GENDER  831200 non-null  object
dtypes: object(2)
memory usage: 12.7+ MB


In [53]:
df_latam.duplicated().value_counts()

False    831200
Name: count, dtype: int64

In [54]:
# We will perform an aditional cleaning on the data sets for NAME column
# in this step we seek avoint tildes and normalize the text name

def normalize_text(text:str):
    if not isinstance(text, str):
        return ""
    # convert to lower and remove spaces
    text = text.lower().strip()
    # remove tildes
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8')
    return text

In [55]:
df_latam['NAME_CLEAN'] = df_latam['NAME'].apply(normalize_text)
df_latam

,NAME,GENDER,NAME_CLEAN
0,Elena,F,elena
1,Anyi,F,anyi
2,Maura,F,maura
3,Yamilet,F,yamilet
4,Joana,F,joana
...,...,...,...
831195,Joseylorena,F,joseylorena
831196,Mar Let,F,mar let
831197,Nixe,F,nixe
831198,Manya,M,manya


In [56]:
# We validate duplicated values after name normalization
df_latam.duplicated(subset=["GENDER", "NAME_CLEAN"]).value_counts()

False    781516
True      49684
Name: count, dtype: int64

In [57]:
df_latam.drop_duplicates(subset=["GENDER", "NAME_CLEAN"], inplace=True)

In [58]:
print(df_latam.columns)

Index(['NAME', 'GENDER', 'NAME_CLEAN'], dtype='object')


In [59]:
df_latam.duplicated(subset=["GENDER", "NAME_CLEAN"]).value_counts()

False    781516
Name: count, dtype: int64

In [60]:
df_latam = df_latam[["NAME_CLEAN", "GENDER"]]

In [61]:
# Double check on gender clases
df_latam['GENDER'].value_counts()

GENDER
F    392740
M    388776
Name: count, dtype: int64

#### Results after data normalization

After data normalization and duplicated removing, we got 781,516 records where 392,740 are as female and 388,776 are as male

## 2. Strating data preparation for NN model

In [62]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib

In [63]:
df_latam.head(10)

,NAME_CLEAN,GENDER
0,elena,F
1,anyi,F
2,maura,F
3,yamilet,F
4,joana,F
5,amed,M
6,yohan,M
7,gisel,F
8,donald,M
9,la peque,F


In [64]:
df = df_latam.copy()

In [65]:
# encode gender columna with label enconder

le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['GENDER'])

# Saving the encoder to revert prediction
joblib.dump(le, '../data/gold/model/gender_encoder.pkl')

['../data/gold/model/gender_encoder.pkl']

In [66]:
df.head(5)

,NAME_CLEAN,GENDER,gender_encoded
0,elena,F,0
1,anyi,F,0
2,maura,F,0
3,yamilet,F,0
4,joana,F,0


In [67]:

tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(df['NAME_CLEAN'])


sequences = tokenizer.texts_to_sequences(df['NAME_CLEAN'])


X = pad_sequences(sequences, maxlen=20, padding='post')

joblib.dump(tokenizer, '../data/gold/model/names_encoder.pkl')

df['name_encoded'] = list(X)

In [68]:
v_size = len(tokenizer.word_index)
v_size

29

In [69]:
df.head()

,NAME_CLEAN,GENDER,gender_encoded,name_encoded
0,elena,F,0,"[3, 6, 3, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,anyi,F,0,"[1, 5, 15, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2,maura,F,0,"[12, 1, 14, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
3,yamilet,F,0,"[15, 1, 12, 2, 6, 3, 11, 0, 0, 0, 0, 0, 0, 0, ..."
4,joana,F,0,"[17, 7, 1, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


In [70]:
df_encoded = df[["name_encoded", "gender_encoded"]].reset_index()
df_encoded.count()

index             781516
name_encoded      781516
gender_encoded    781516
dtype: int64

## 3. Split dataframe on train 60%, validation 20% and test 20% data sets

In [71]:
from sklearn.model_selection import train_test_split

# Names encoded
x = df_encoded["name_encoded"]
# Gender encoded
y = df_encoded["gender_encoded"]

# Set Train data set
X_train, X_tmp, Y_train, Y_tmp = train_test_split(
    x,y, test_size=0.4, random_state=42
)

# Set test and validation data set
X_val, X_test, Y_val, Y_test = train_test_split(
    X_tmp, Y_tmp, test_size=0.5, random_state=42
)


In [72]:
# Save train data frames on gold stage

train = pd.concat([X_train, Y_train], axis=1)
val = pd.concat([X_val, Y_val], axis=1)
test = pd.concat([X_test, Y_test], axis=1)

train.to_parquet("../data/gold/model/mane_gender_encoded_train.parquet", index=False)
val.to_parquet("../data/gold/model/mane_gender_encoded_val.parquet", index=False)
test.to_parquet("../data/gold/model/mane_gender_encoded_test.parquet", index=False)